In [1]:
import sys, os
import re
import matplotlib.pyplot as plt
import seaborn as sns
import gc
import glob
import pandas as pd
import itertools
import ast
import numpy as np
from pathlib import Path

In [2]:
from tqdm.notebook import tqdm

tqdm.pandas()

### Loading package

In [3]:
import sys
from pathlib import Path

here_path = Path().resolve()
repo_path = here_path.parents[1]
sys.path.append(str(repo_path))

In [4]:
from py.utils import verifyDir,verifyFile

In [5]:
from py.config import Config

cfg = Config()

np.random.seed(cfg.RANDOM_STATE)
cfg.DATA_PATH, cfg.MODEL_PATH

('/media/felipe/DATA21/datasets/', '/media/felipe/DATA21/models/')

### Loading data

In [6]:
DATA_PATH=f"{cfg.DATA_PATH}crimebb/"
CSV_PATH = f"{DATA_PATH}/{cfg.YEAR}/csv/"
CSV_PROCESSED = f"{CSV_PATH}/websites/"
HF_PROCESSED = f"{DATA_PATH}/{cfg.YEAR}/csv/topics/{cfg.TOPIC_SEARCH}/"

In [7]:
verifyDir(HF_PROCESSED)

In [8]:
%%time
hf_reader = pd.read_csv(f"{HF_PROCESSED}HackForums_CVE_Posts.csv", sep='\t', low_memory=False, iterator=True)
            
hf_df = pd.DataFrame()

len_readed=cfg.CHUNK_SIZE
while len_readed>=cfg.CHUNK_SIZE:
    current_df = hf_reader.get_chunk(cfg.CHUNK_SIZE).copy()
    len_readed = current_df.shape[0]
    current_df.drop_duplicates(inplace=True)
    
    #current_df = current_df[["site_name", "board_title", "thread_id", "thread_title", "content", "post_data_creation"]].copy().drop_duplicates()
    current_df = current_df[~current_df["content"].isna()].copy()
    current_df = current_df[~current_df["thread_title"].isna()].copy()
    current_df.drop_duplicates(inplace=True)
    
    current_df["board_title"] = current_df["board_title"].astype(str)
    current_df["thread_title"] = current_df["thread_title"].astype(str)
    current_df["content"] = current_df["content"].astype(str)
    current_df["cve_codes_list"] = current_df["cve_codes_list"].apply(ast.literal_eval)
    
    hf_df = pd.concat([hf_df, current_df], ignore_index=True)
    
    del current_df
    gc.collect()

CPU times: user 212 ms, sys: 24.9 ms, total: 237 ms
Wall time: 236 ms


### Summarizing data by threads and CVE

In [9]:
%%time
hf_threads_df = pd.pivot_table(hf_df,
                           index=["site_id", "site_name", "board_id", "board_title", "thread_id", "thread_title"], 
                           values=["content", "cve_codes_list"],
                           aggfunc={
                               "content": lambda x: cfg.KEYWORD_SEP.join(list(x)),
                               "cve_codes_list": lambda x: list(itertools.chain(*x)),
                           }).reset_index(drop=False)
hf_threads_df

CPU times: user 82.4 ms, sys: 3.01 ms, total: 85.5 ms
Wall time: 84.9 ms


,site_id,site_name,board_id,board_title,thread_id,thread_title,content,cve_codes_list
0,0,hackforums.net,2,"Rules, Announcements, News, and Feedback",371099,Rep Explained to new members.,***CITING***[https://hackforums.net/showthread...,[cve]
1,0,hackforums.net,2,"Rules, Announcements, News, and Feedback",955196,Contest win 15$/Upgrade,Post:18\nNumber:543\nPaypal Account:djurre@zen...,[cve]
2,0,hackforums.net,2,"Rules, Announcements, News, and Feedback",1727141,The Logo Competition - Win prizes!,***IFRAME***[//www.youtube.com/embed/xfcVEh5KW...,[cve]
3,0,hackforums.net,2,"Rules, Announcements, News, and Feedback",1736219,gz hashy you won my account! have fun,26 Please!\n\nThanks in advancve\n\nThanks!,[cve]
4,0,hackforums.net,2,"Rules, Announcements, News, and Feedback",2218569,"GloryCraft $100,000 Contest",Gratz Mcveg! 1 over mine u bitch 0.0 XD FAMVEE...,"[cve, cve, cve]"
...,...,...,...,...,...,...,...,...
3584,0,hackforums.net,374,Premium Tools and Programs,6005894,[NEW] SILENT OFFICE EXPLOIT | EXE TO DOC | WOR...,"Lot, CVE not even mentioned. Probably a scam -...","[cve, cve, cve, cve, cve]"
3585,0,hackforums.net,374,Premium Tools and Programs,6011940,>>>>> [NEW] SILENT OFFICE EXPLOIT | .DOC | GMA...,Silent Office Exploits\n\n\nDEMO\n\n\n***IFRAM...,[cve-2017-11882]
3586,0,hackforums.net,374,Premium Tools and Programs,6015899,"RDP Exploit, daily you get more than 20 RDPs U...",in the sales topic you must indicate the vulne...,[cve]
3587,0,hackforums.net,374,Premium Tools and Programs,6016812,SUDO EXPLOIT | LINUX SUDO BYPASS | DL & EXEC |...,About the CVE:\n\nThe CVE-2019-14287 vulnerabi...,"[cve-2019-14287, cve, cve-2019-14287, cve, cve..."


In [10]:
%%time
hf_threads_df["cve_codes_len"] = hf_threads_df["cve_codes_list"].apply(lambda x: len(x) )
hf_threads_df["cve_unique_list"] = hf_threads_df["cve_codes_list"].apply(lambda x: list(set(itertools.chain(x))))
hf_threads_df["cve_unique_len"] = hf_threads_df["cve_unique_list"].apply(lambda x: len(x) )
hf_threads_df.sort_values(by="cve_unique_len", ascending=False, inplace=True)

CPU times: user 7.1 ms, sys: 3 μs, total: 7.1 ms
Wall time: 6.43 ms


In [11]:
hf_threads_df = hf_threads_df[hf_threads_df["cve_codes_len"]>0].copy()
hf_threads_df

,site_id,site_name,board_id,board_title,thread_id,thread_title,content,cve_codes_list,cve_codes_len,cve_unique_list,cve_unique_len
178,0,hackforums.net,4,Beginner Hacking,4299925,+Pentesting: Anyone have a list of vulnerable ...,***CITING***[https://hackforums.net/showthread...,"[cve-2014-0323, cve-2014-0317, cve-2014-0315, ...",62,"[cve-2013-3900, cve-2013-3181, cve-2013-1313, ...",59
553,0,hackforums.net,13,Computer Protection and Security Alerts,4176757,[WARNING] Update your Java installation - j8u5...,***IMG***[https://i.imgur.com/kxTfXwL.png]***I...,"[cve-, cve-2013-6629, cve-2013-6954, cve-2014-...",38,"[cve-2014-0454, cve-2014-2409, cve-2014-2403, ...",38
2082,0,hackforums.net,110,"White Hat Malware, Virus, and Rat Removal Help",3851963,Possible infection?,***LINK***https://hackforums.net/showthread.ph...,"[cve-2008-5499, cve-2008-5499, cve-2010-0480, ...",77,"[cve-2010-0842, cve-2010-0480, cve-2010-3563, ...",36
1682,0,hackforums.net,92,"Botnets, IRC Bots, and Zombies",5490019,Terror Exploit Kit [DUMP],Yes i am the original owner of terror exploit ...,"[cve-2014-6332, cve, cve, cve, cve, cve, cve, ...",54,"[cve-2014-8636, cve-2015-2419, cve-2012-1875, ...",30
3195,0,hackforums.net,231,Pentesting and Forensics,4083645,[Tutorial] ~ Writing TCP Bind Shell in Linux A...,***CITING***[https://hackforums.net/showthread...,"[cve-2015-3113, cve-2015-2717, cve-2015-1360, ...",28,"[cve-2015-1360, cve-2015-1744, cve-2015-3049, ...",22
...,...,...,...,...,...,...,...,...,...,...,...
1312,0,hackforums.net,48,Requests for Hacking,4534441,Drupal 6.2.4 exploit,Take a look threw cvedetails you might find so...,"[cve, cve, cve]",3,[cve],1
1315,0,hackforums.net,48,Requests for Hacking,4691659,SHA512 hash crack?,"Hi,\n\nI'm not skilled with any of this stuff ...",[cve],1,[cve],1
1316,0,hackforums.net,48,Requests for Hacking,4762407,Joomla,cvedetails.com/vulnerability-list/vendor_id-34...,[cve],1,[cve],1
1317,0,hackforums.net,48,Requests for Hacking,4815872,Request for this shit website.,"That GPA shit is running Apache 2.4.10, which ...","[cve, cve]",2,[cve],1


### Filtering references

***IMG*** (URL to an image); 

***CITING*** (URL to a cited post); 

***IFRAME*** (URL to an external iframe, usually used for embedding videos); 

***LINK*** (Other external URL) 

***CODE*** (Source code written explicitly); 

***ATTACHMENT*** (Link to an attached asset); 

***QUOTE*** (Quote to an external source i.e., this annotation omits quotation of other posts since these are annotated as cited posts).

crime_type, intent, and posttype where labeled in hackforum and then infered for the remaining posts.

In [12]:
from py.utils import filter_attachment, replace_attachment

In [13]:
ref_types = ["***IMG***", "***CITING***", "***IFRAME***", "***LINK***", "***CODE***", "***ATTACHMENT***", "***QUOTE***"]

In [14]:
for pattern in ref_types:
    hf_threads_df[f'r_{pattern.replace("*", "")}'] = hf_threads_df["content"].apply(lambda x: filter_attachment(x, pattern))

In [15]:
hf_threads_df["content_orig"] = hf_threads_df["content"].copy()
hf_threads_df["content"] = hf_threads_df["content"].apply( lambda x: replace_attachment(x, ref_types) )

In [16]:
hf_threads_df.info()

<class 'pandas.DataFrame'>
Index: 3589 entries, 178 to 1302
Data columns (total 19 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   site_id          3589 non-null   int64 
 1   site_name        3589 non-null   str   
 2   board_id         3589 non-null   int64 
 3   board_title      3589 non-null   str   
 4   thread_id        3589 non-null   int64 
 5   thread_title     3589 non-null   str   
 6   content          3589 non-null   str   
 7   cve_codes_list   3589 non-null   object
 8   cve_codes_len    3589 non-null   int64 
 9   cve_unique_list  3589 non-null   object
 10  cve_unique_len   3589 non-null   int64 
 11  r_IMG            3589 non-null   object
 12  r_CITING         3589 non-null   object
 13  r_IFRAME         3589 non-null   object
 14  r_LINK           3589 non-null   object
 15  r_CODE           3589 non-null   object
 16  r_ATTACHMENT     3589 non-null   object
 17  r_QUOTE          3589 non-null   object
 18  co

In [17]:
hf_threads_df.to_csv(f"{HF_PROCESSED}HackForums_CVE_Threads.csv", sep='\t', index=False)

## Statictics

In [18]:
len(hf_threads_df["site_name"].unique()), len(hf_threads_df["board_title"].unique()), len(hf_threads_df["thread_title"].unique()), len(hf_threads_df["content"].unique())

(1, 140, 3534, 3487)

In [19]:
unique_cve_codes = list(set(itertools.chain(*hf_threads_df["cve_unique_list"].values)))
len(unique_cve_codes)#, unique_cve_codes

1174

### Extra

In [20]:
len(hf_threads_df["thread_id"].unique()), len(hf_threads_df["thread_title"].unique())

(3589, 3534)

In [21]:
hf_threads_df[hf_threads_df["thread_title"]=="[TUT] How to run Exploit Scripts! [TUT]"]

,site_id,site_name,board_id,board_title,thread_id,thread_title,content,cve_codes_list,cve_codes_len,cve_unique_list,cve_unique_len,r_IMG,r_CITING,r_IFRAME,r_LINK,r_CODE,r_ATTACHMENT,r_QUOTE,content_orig
2512,0,hackforums.net,129,Python,318295,[TUT] How to run Exploit Scripts! [TUT],In this TuT im going to show you how to run th...,[cve-2007-1531],1,[cve-2007-1531],1,"[[images/smilies/biggrin.gif], [images/smilies...",[],[],[],[[#!/usr/bin/env python]\n#\n# :: Kristian Her...,[],[],In this TuT im going to show you how to run th...
1198,0,hackforums.net,47,Hacking Tutorials,318308,[TUT] How to run Exploit Scripts! [TUT],In this TuT im going to show you how to run th...,[cve-2007-1531],1,[cve-2007-1531],1,"[[images/smilies/biggrin.gif], [images/smilies...",[],[],[],[[#!/usr/bin/env python]\n#\n# :: Kristian Her...,[],[],In this TuT im going to show you how to run th...


In [22]:
hf_threads_df[(hf_threads_df["board_title"]=="Beginner Hacking") & (hf_threads_df["thread_title"]=="Exploits")]

,site_id,site_name,board_id,board_title,thread_id,thread_title,content,cve_codes_list,cve_codes_len,cve_unique_list,cve_unique_len,r_IMG,r_CITING,r_IFRAME,r_LINK,r_CODE,r_ATTACHMENT,r_QUOTE,content_orig
124,0,hackforums.net,4,Beginner Hacking,3270349,Exploits,Guys can you help me on exploits?\nI have foun...,[cve-2010-0425],1,[cve-2010-0425],1,[],[],[],[http://exploitsdownload.com/search?q=%09...0-...,[],[],[],Guys can you help me on exploits?\nI have foun...
186,0,hackforums.net,4,Beginner Hacking,4462478,Exploits,I am not interested in exploits but I got an e...,[cve-2014-6271],1,[cve-2014-6271],1,[],[],[],[http://web.nvd.nist.gov/view/vuln/detail...zr...,[],[],[],I am not interested in exploits but I got an e...


## Comparing previous version

Definitions and labeling Rules:

* PoC:
  The proof of concept label refers to vulnerabilities in a non-threatful way. In these threads usually we saw discussions that don't show signs of future exploitation, or talks about exploiting in a controlled environment, are considered to be PoC. Thus, this class represents the less dangerous threads.
  keywords:
   - trillium
   - poc
   - tutorial
   - penetration test
   - guide
   - etc

* Weaponization and trading:
  The Weaponization and trading label refers to discussions that can lead to an exploitation but are not there yet. e.g. In these threads users are very close to succeed in hacking, but something in the code doesn't work, so they share their code asking for help. In others, offers money to do it. Colloquially, we can think of this class as the "almost there" threads.
  keywords: 
   - fud (fully undetectable)
   - money, price, bitcoin
   - buy, sell, offer
   - packs
   - metasploit
   - sharing of code of someone trying to hack
   - etc

* Exploitation:
  The Exploitation label refers to exploit or cite vulnerabilities. In these threads we see confirmed trades and exploitations by users. This class, represents the most dangerous threads which share ways to attack vulnerabilities.
  keywords: 
   - Sell, buy or trade confirmed
   - pishing
   - man-in-the-middle
   - attack
   - etc


* Help:
  The help label refers to users asking for help, guidelines, tutorials, etc.
  keywords: 
   - help
   - need
   - trying
   - looking
   - etc

### Previous Threads

https://docs.google.com/spreadsheets/d/117cCExWGNJVqI3_UY_nLSdCcnwP5VdXovmqpjRh4YNk

In [23]:
cc_df = pd.read_csv(f"{CSV_PATH}/HF_annotations.csv", sep=",", low_memory=False)
cc_df.drop(columns=["1193", "threads refer to CVE"], inplace=True)
cc_df.rename(columns={"IdThread":"thread_id", "Label":"labels"}, inplace=True)
cc_df.reset_index(inplace=True, drop=True)
cc_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1193 entries, 0 to 1192
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   thread_id  1193 non-null   int64
 1   labels     1067 non-null   str  
dtypes: int64(1), str(1)
memory usage: 27.8 KB


In [24]:
print("total", cc_df["labels"].value_counts().sum())
print("Nan", len(cc_df[cc_df["labels"].isna()]))
cc_df["labels"].value_counts()

total 1067
Nan 126


labels
weaponization    410
poc              247
others           195
exploitation     107
warning           55
help              43
scam              10
Name: count, dtype: int64

### Join to get threads and labels

In [25]:
hf_threads_labels_df = pd.merge(cc_df[~cc_df["labels"].isna()].copy(), hf_threads_df, on="thread_id", how="left")
hf_threads_labels_df = hf_threads_labels_df[~hf_threads_labels_df["content"].isna()].copy()
hf_threads_labels_df.reset_index(inplace=True, drop=True)
hf_threads_labels_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1067 entries, 0 to 1066
Data columns (total 20 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   thread_id        1067 non-null   int64 
 1   labels           1067 non-null   str   
 2   site_id          1067 non-null   int64 
 3   site_name        1067 non-null   str   
 4   board_id         1067 non-null   int64 
 5   board_title      1067 non-null   str   
 6   thread_title     1067 non-null   str   
 7   content          1067 non-null   str   
 8   cve_codes_list   1067 non-null   object
 9   cve_codes_len    1067 non-null   int64 
 10  cve_unique_list  1067 non-null   object
 11  cve_unique_len   1067 non-null   int64 
 12  r_IMG            1067 non-null   object
 13  r_CITING         1067 non-null   object
 14  r_IFRAME         1067 non-null   object
 15  r_LINK           1067 non-null   object
 16  r_CODE           1067 non-null   object
 17  r_ATTACHMENT     1067 non-null   object
 18 

In [26]:
print("total", hf_threads_labels_df["labels"].value_counts().sum() + len(hf_threads_labels_df[hf_threads_labels_df["labels"].isna()]))
print("Nan", len(hf_threads_labels_df[hf_threads_labels_df["labels"].isna()]))
hf_threads_labels_df["labels"].value_counts()

total 1067
Nan 0


labels
weaponization    410
poc              247
others           195
exploitation     107
warning           55
help              43
scam              10
Name: count, dtype: int64

## Statistics

In [27]:
len(hf_threads_labels_df["site_name"].unique()), len(hf_threads_labels_df["board_title"].unique()), len(hf_threads_labels_df["thread_title"].unique()), len(hf_threads_labels_df["content"].unique())

(1, 67, 1053, 1058)

In [28]:
unique_cve_codes = list(set(itertools.chain(*hf_threads_labels_df["cve_unique_list"].values)))
len(unique_cve_codes) #, hf_threads_cve_codes, 

1028

#### Total threads in HackForums with CVE before join

In [29]:
hf_cve_threads_count_df = pd.pivot_table(hf_threads_df, 
              index=["board_title"], 
              values=["thread_id"],
              aggfunc={
                  "thread_id": lambda x: len(list(set(x))),
              }).reset_index(drop=False).rename(columns={"thread_id": "num_thread"}).sort_values(by="num_thread", ascending=False)
hf_cve_threads_count_df.head(10)

,board_title,num_thread
9,Beginner Hacking,321
84,Pentesting and Forensics,298
131,Website and Forum Hacking,261
118,The Lounge,239
90,Premium Sellers Section,175
10,"Botnets, IRC Bots, and Zombies",148
52,Hacking Tools and Programs,141
39,Free Services and Giveaways,107
105,Secondary Sellers Market,91
79,News and Happenings,88


#### Total Threads in HackForums after join and concat

In [30]:
hf_boar_threads_count_df = pd.pivot_table(hf_threads_labels_df, 
              index=["board_title"], 
              values=["thread_id"],
              aggfunc={
                  "thread_id": len,
              }).reset_index(drop=False).rename(columns={"thread_id": "num_thread"}).sort_values(by="num_thread", ascending=False)
hf_boar_threads_count_df.head(10)

,board_title,num_thread
40,Pentesting and Forensics,151
5,Beginner Hacking,134
61,Website and Forum Hacking,114
42,Premium Sellers Section,64
6,"Botnets, IRC Bots, and Zombies",59
24,Hacking Tools and Programs,51
38,News and Happenings,45
62,"White Hat Malware, Virus, and Rat Removal Help",39
25,Hacking Tutorials,30
49,Secondary Sellers Market,28


#### Total CVEs

In [31]:
cve_board_df = pd.pivot_table(hf_threads_labels_df, 
              index=["site_name", "board_title", "thread_title"], 
              values=["cve_codes_len", "cve_unique_len"],
              aggfunc={
                  "cve_codes_len": "sum",
                  "cve_unique_len": "sum",
              }).reset_index(drop=False).sort_values(by="cve_unique_len", ascending=False)
cve_board_df.head(10)

,site_name,board_title,thread_title,cve_codes_len,cve_unique_len
13,hackforums.net,Beginner Hacking,+Pentesting: Anyone have a list of vulnerable ...,62,59
265,hackforums.net,Computer Protection and Security Alerts,[WARNING] Update your Java installation - j8u5...,38,38
1008,hackforums.net,"White Hat Malware, Virus, and Rat Removal Help",Possible infection?,77,36
184,hackforums.net,"Botnets, IRC Bots, and Zombies",Terror Exploit Kit [DUMP],54,30
637,hackforums.net,Pentesting and Forensics,[Tutorial] ~ Writing TCP Bind Shell in Linux A...,28,22
832,hackforums.net,Secondary Sellers Market,bleeding life v2:reloaded+crimepack 3.1.3 work...,27,19
999,hackforums.net,"White Hat Malware, Virus, and Rat Removal Help",I am infected,36,18
254,hackforums.net,Computer Protection and Security Alerts,Update for Adobe software (12-13-2016),19,18
693,hackforums.net,Premium Sellers Section,RIG exploit kit Official HF Sales thread $30 D...,16,16
472,hackforums.net,News and Happenings,Fancy Bear part 5: ESET Report,16,15


### Saving data

In [32]:
hf_threads_labels_df.drop(columns=["cve_unique_len", "cve_codes_len"], inplace=True)
hf_threads_labels_df.rename(columns={"labels": "annotations"}, inplace=True)
hf_threads_labels_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1067 entries, 0 to 1066
Data columns (total 18 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   thread_id        1067 non-null   int64 
 1   annotations      1067 non-null   str   
 2   site_id          1067 non-null   int64 
 3   site_name        1067 non-null   str   
 4   board_id         1067 non-null   int64 
 5   board_title      1067 non-null   str   
 6   thread_title     1067 non-null   str   
 7   content          1067 non-null   str   
 8   cve_codes_list   1067 non-null   object
 9   cve_unique_list  1067 non-null   object
 10  r_IMG            1067 non-null   object
 11  r_CITING         1067 non-null   object
 12  r_IFRAME         1067 non-null   object
 13  r_LINK           1067 non-null   object
 14  r_CODE           1067 non-null   object
 15  r_ATTACHMENT     1067 non-null   object
 16  r_QUOTE          1067 non-null   object
 17  content_orig     1067 non-null   str   
dtyp

In [33]:
hf_threads_labels_df.to_csv(f"{HF_PROCESSED}HackForums_CVE_Threads_and_Labels.csv", sep='\t', index=False)